# CSE6242 - HW3 - Q1

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> remove any comment that says "#export" because that will crash the autograder in Gradescope. We use this comment to export your code in these cells for grading.
</div>

Pyspark Imports

In [1]:
#export
### DO NOT MODIFY THIS CELL ###
import pyspark
from pyspark.sql import SQLContext
from pyspark.sql.functions import hour, when, col, date_format, to_timestamp, ceil, coalesce

Initialize PySpark Context

In [2]:
### DO NOT MODIFY THIS CELL ###
sc = pyspark.SparkContext(appName="HW3-Q1")
sqlContext = SQLContext(sc)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/25 00:57:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
/usr/local/lib/python3.9/dist-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Define function for loading data

In [3]:
### DO NOT MODIFY THIS CELL ###
def load_data():
    df = sqlContext.read.option("header",True) \
     .csv("yellow_tripdata_2019-01_short.csv")
    return df

### Q1.1

Perform data casting to clean incoming dataset

In [4]:
#export

# define how to access the columns and target types
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, FloatType, TimestampType

def clean_data(df):
    '''
    input: df a dataframe
    output: df a dataframe with the all the original columns
    '''
    
    # START YOUR CODE HERE ---------
    vendor_id_name = 'VendorID'
    tpep_pickup_datetime_name = 'tpep_pickup_datetime'
    tpep_dropoff_datetime_name = 'tpep_dropoff_datetime'
    passenger_count_name = 'passenger_count'
    trip_distance_name = 'trip_distance'
    rate_code_id_name = 'RatecodeID'
    store_and_fwd_flag_name = 'store_and_fwd_flag'
    pu_location_id_name = 'PULocationID'
    do_location_id_name = 'DOLocationID'
    payment_type_name = 'payment_type'
    fare_amount_name = 'fare_amount'
    extra_name = 'extra'
    mta_tax_name = 'mta_tax'
    tip_amount_name = 'tip_amount'
    tolls_amount_name = 'tolls_amount'
    improvement_surcharge_name = 'improvement_surcharge'
    total_amount_name = 'total_amount'
    congestion_surcharge_name = 'congestion_surcharge'
    # Incoming columns
    # root
    #  |-- VendorID: string (nullable = true)
    #  |-- tpep_pickup_datetime: string (nullable = true)
    #  |-- tpep_dropoff_datetime: string (nullable = true)
    #  |-- passenger_count: string (nullable = true)
    #  |-- trip_distance: string (nullable = true)
    #  |-- RatecodeID: string (nullable = true)
    #  |-- store_and_fwd_flag: string (nullable = true)
    #  |-- PULocationID: string (nullable = true)
    #  |-- DOLocationID: string (nullable = true)
    #  |-- payment_type: string (nullable = true)
    #  |-- fare_amount: string (nullable = true)
    #  |-- extra: string (nullable = true)
    #  |-- mta_tax: string (nullable = true)
    #  |-- tip_amount: string (nullable = true)
    #  |-- tolls_amount: string (nullable = true)
    #  |-- improvement_surcharge: string (nullable = true)
    #  |-- total_amount: string (nullable = true)
    #  |-- congestion_surcharge: string (nullable = true)

    # Casting columns
    # total_amount — float
    # tip_amount — float
    # trip_distance — float
    # fare_amount — float
    # tpep_pickup_datetime — timestamp
    # tpep_dropoff_datetime — timestamp
    
    df = df.withColumn(passenger_count_name, col(passenger_count_name).cast(IntegerType())) \
            .withColumn(total_amount_name, col(total_amount_name).cast(FloatType())) \
            .withColumn(tip_amount_name, col(tip_amount_name).cast(FloatType())) \
            .withColumn(trip_distance_name, col(trip_distance_name).cast(FloatType())) \
            .withColumn(fare_amount_name, col(fare_amount_name).cast(FloatType())) \
            .withColumn(tpep_pickup_datetime_name, col(tpep_pickup_datetime_name).cast(TimestampType())) \
            .withColumn(tpep_dropoff_datetime_name, col(tpep_dropoff_datetime_name).cast(TimestampType()))

    # END YOUR CODE HERE -----------

    return df

### Q1.2

Find rate per person for based on how many passengers travel between pickup and dropoff locations. 

In [5]:
#export
from pyspark.sql import functions as F

def common_pair(df):
    '''
    input: df a dataframe
    output: df a dataframe with following columns:
            - PULocationID
            - DOLocationID
            - total_passenger_count
            - per_person_rate
            
    per_person_rate is the total_amount per person for a given pair.
    
    '''
    
    # START YOUR CODE HERE ---------
    vendor_id_name = 'VendorID'
    tpep_pickup_datetime_name = 'tpep_pickup_datetime'
    tpep_dropoff_datetime_name = 'tpep_dropoff_datetime'
    passenger_count_name = 'passenger_count'
    trip_distance_name = 'trip_distance'
    rate_code_id_name = 'RatecodeID'
    store_and_fwd_flag_name = 'store_and_fwd_flag'
    pu_location_id_name = 'PULocationID'
    do_location_id_name = 'DOLocationID'
    payment_type_name = 'payment_type'
    fare_amount_name = 'fare_amount'
    extra_name = 'extra'
    mta_tax_name = 'mta_tax'
    tip_amount_name = 'tip_amount'
    tolls_amount_name = 'tolls_amount'
    improvement_surcharge_name = 'improvement_surcharge'
    total_amount_name = 'total_amount'
    congestion_surcharge_name = 'congestion_surcharge'
    # You will be modifying the function common_pair. Return the top 10 pickup-dropoff location pairs that have
    # the highest sum of passenger_count who have traveled between them. Sort the location pairs by total passengers
    # between pairs. For each location pair, also compute the average amount per passenger over all trips (name this
    # per_person_rate), utilizing total_amount.
    # For pairs with the same total passengers, sort them in descending order of per_person_rate. Filter out any trips
    # that have the same pick-up and drop-off location.
    # Rename the column for total passengers to total_passenger_count.

    # Incoming
    # root
    #  |-- VendorID: string (nullable = true)
    #  |-- tpep_pickup_datetime: timestamp (nullable = true)
    #  |-- tpep_dropoff_datetime: timestamp (nullable = true)
    #  |-- passenger_count: string (nullable = true)
    #  |-- trip_distance: float (nullable = true)
    #  |-- RatecodeID: string (nullable = true)
    #  |-- store_and_fwd_flag: string (nullable = true)
    #  |-- PULocationID: string (nullable = true)
    #  |-- DOLocationID: string (nullable = true)
    #  |-- payment_type: string (nullable = true)
    #  |-- fare_amount: float (nullable = true)
    #  |-- extra: string (nullable = true)
    #  |-- mta_tax: string (nullable = true)
    #  |-- tip_amount: float (nullable = true)
    #  |-- tolls_amount: string (nullable = true)
    #  |-- improvement_surcharge: string (nullable = true)
    #  |-- total_amount: float (nullable = true)
    #  |-- congestion_surcharge: string (nullable = true)

    # The initial filtering criteria is top 10 pu/do ID locations
    total_passenger_count_name = 'total_passenger_count'
    row_limit = 10
    # And we want the total fare so we can compute an avg on next step
    total_total_amount_name = 'total_total_amount'
    
    df = df \
        .filter(F.col(pu_location_id_name) != F.col(do_location_id_name)) \
        .groupBy(pu_location_id_name, do_location_id_name).agg(
            F.sum(passenger_count_name).alias(total_passenger_count_name),
            F.sum(total_amount_name).alias(total_total_amount_name)
        )
    
    per_person_rate_name = 'per_person_rate'
    df = df.withColumn(per_person_rate_name, df[total_total_amount_name] / df[total_passenger_count_name])

    df = df.orderBy(
        F.desc(total_passenger_count_name),
        F.desc(per_person_rate_name)
    ).limit(row_limit)

    df = df.select(pu_location_id_name, do_location_id_name, total_passenger_count_name, per_person_rate_name)
    # END YOUR CODE HERE -----------
    
    return df

### Q1.3

Find trips which trip distances generate the highest tip percentage.

In [16]:
#export
from pyspark.sql import functions as F

def distance_with_most_tip(df):
    '''
    input: df a dataframe
    output: df a dataframe with following columns:
            - trip_distance
            - tip_percent
            
    trip_percent is the percent of tip out of fare_amount
    
    '''
    
    # START YOUR CODE HERE ---------
    vendor_id_name = 'VendorID'
    tpep_pickup_datetime_name = 'tpep_pickup_datetime'
    tpep_dropoff_datetime_name = 'tpep_dropoff_datetime'
    passenger_count_name = 'passenger_count'
    trip_distance_name = 'trip_distance'
    rate_code_id_name = 'RatecodeID'
    store_and_fwd_flag_name = 'store_and_fwd_flag'
    pu_location_id_name = 'PULocationID'
    do_location_id_name = 'DOLocationID'
    payment_type_name = 'payment_type'
    fare_amount_name = 'fare_amount'
    extra_name = 'extra'
    mta_tax_name = 'mta_tax'
    tip_amount_name = 'tip_amount'
    tolls_amount_name = 'tolls_amount'
    improvement_surcharge_name = 'improvement_surcharge'
    total_amount_name = 'total_amount'
    congestion_surcharge_name = 'congestion_surcharge'

    tip_percent_name = 'tip_percent'
    tip_limit = 15
    # Filter the data for trips having fares (fare_amount) greater than $2.00 and a trip distance (trip_distance) greater
    # than 0. Calculate the tip percent (tip_amount * 100 / fare_amount) for each trip. Round all trip distances up
    # to the closest mile and find the average tip_percent for each trip_distance. Sort the result in descending
    # order of tip_percent to obtain the top 15 trip distances which tip the most generously. Rename the column for
    # rounded trip distances to trip_distance, and the column for average tip percents tip_percent.
    df = df.filter((F.col(fare_amount_name) > 2.0) & (F.col(trip_distance_name) > 0.0))
    df = df.withColumn(tip_percent_name, (100.0 * df[tip_amount_name]) / df[fare_amount_name])
    df = df.withColumn(trip_distance_name, F.round(df[trip_distance_name], 0))
    df = df.groupBy(trip_distance_name).agg(F.avg(tip_percent_name).alias(tip_percent_name))
    df = df.orderBy(
        F.desc(tip_percent_name)
    ).limit(tip_limit)
    # END YOUR CODE HERE -----------
    
    return df

### Q1.4

Determine the average speed at different times of day.

In [7]:
#export
def time_with_most_traffic(df):
    '''
    input: df a dataframe
    output: df a dataframe with following columns:
            - time_of_day
            - am_avg_speed
            - pm_avg_speed
            
    am_avg_speed and pm_avg_speed are the average trip distance / average trip time calculated for each hour
    
    '''
    
    # START YOUR CODE HERE ---------
    define_names()
    # END YOUR CODE HERE -----------
    
    return df

## The below cells are for you to investigate your solutions and will not be graded

In [8]:
#import IPython

# This will immediately restart the kernel
#IPython.Application.instance().kernel.do_shutdown(True)

In [9]:
#df = load_data()
#df = clean_data(df)
#df.printSchema()

In [10]:
#####  ABOVE NEEDS TO BE PURGED

In [11]:
df = load_data()
df = clean_data(df)
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|              1|          1.5|         1|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.65|           0|                  0.3

In [12]:
df = load_data()
common_pair(df).show()

+------------+------------+---------------------+-----------------+
|PULocationID|DOLocationID|total_passenger_count|  per_person_rate|
+------------+------------+---------------------+-----------------+
|         239|         238|                 62.0|4.262741935483872|
|         237|         236|                 60.0|4.482500000000002|
|         263|         141|                 52.0|3.419038461538462|
|         161|         236|                 42.0|5.368571428571429|
|         148|          79|                 42.0|4.711904761904762|
|         142|         238|                 39.0|5.054871794871795|
|         141|         236|                 37.0|4.355675675675675|
|         239|         143|                 37.0|4.252162162162162|
|         239|         142|                 35.0|3.817714285714285|
|          79|         170|                 34.0|6.394705882352943|
+------------+------------+---------------------+-----------------+



In [17]:
df = load_data()
distance_with_most_tip(df).show()

+-------------+------------------+
|trip_distance|       tip_percent|
+-------------+------------------+
|          1.0|16.948128919412067|
|          0.0| 15.42138792497961|
|          2.0|15.316306981406932|
|         17.0| 15.22190379132884|
|          5.0|14.791297026535345|
|          3.0|14.467840960358716|
|         21.0|14.318693182631236|
|         19.0|14.024168214294264|
|          9.0|13.566757643961607|
|          4.0| 13.54864068670642|
|          6.0|13.301328958864719|
|          8.0|11.935883845460518|
|         23.0|11.666666666666666|
|         10.0|11.469710555853517|
|         18.0|11.405847982201076|
+-------------+------------------+



In [ ]:
df = load_data()
time_with_most_traffic(df).show()